# Training Pipeline Walkthrough: Social Media Disaster Dataset

This notebook shows a simple step-by-step flow using:
`ai-ml/datasets/socialmedia-disaster-tweets-DFE.csv`

Goal:
1. Read the new dataset
2. Prepare a clean training file
3. Build config
4. Run pipeline
5. Compare two models


## Step 0: Setup

In [ ]:
from pathlib import Path
import json
import sys

print("[Step 0] Finding project paths...")
TRAINING_ROOT = Path.cwd().resolve()
if not (TRAINING_ROOT / "src").exists():
    for p in [TRAINING_ROOT, *TRAINING_ROOT.parents]:
        candidate = p / "ai-ml" / "training_pipeline"
        if (candidate / "src").exists():
            TRAINING_ROOT = candidate
            break

REPO_ROOT = TRAINING_ROOT.parent.parent

def repo_rel(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(REPO_ROOT.resolve()))
    except Exception:
        return str(path)

DATASET_REL_PATH = Path("..") / "datasets" / "socialmedia-disaster-tweets-DFE.csv"
DATASET_PATH = (TRAINING_ROOT / DATASET_REL_PATH).resolve()

if str(TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINING_ROOT))

print("TRAINING_ROOT:", repo_rel(TRAINING_ROOT))
print("DATASET_REL_PATH (from training_pipeline):", DATASET_REL_PATH.as_posix())
print("DATASET_PATH (from repo root):", repo_rel(DATASET_PATH))
print("DATASET_PATH exists:", DATASET_PATH.exists())


## Step 1: Inspect the raw dataset
Use `latin1` encoding (this file is not UTF-8).

In [ ]:
import pandas as pd

print("[Step 1] Loading raw dataset...")
raw_df = pd.read_csv(DATASET_PATH, encoding="latin1")

print("Shape:", raw_df.shape)
print("Columns:")
for c in raw_df.columns:
    print(" -", c)

print("\nTarget distribution (choose_one):")
print(raw_df["choose_one"].value_counts(dropna=False).head(10))

print("\nSample rows:")
display(raw_df[["choose_one", "choose_one:confidence", "text", "keyword", "location"]].head(5))


## Step 2: Prepare a simple training dataset
We create a binary target and a few numeric features to keep training fast and easy to follow.

In [ ]:
print("[Step 2] Building prepared dataset...")

df = raw_df.copy()
df["label"] = (df["choose_one"].astype(str).str.strip().str.lower() == "relevant").astype(int)
df["text_len"] = df["text"].fillna("").astype(str).str.len()
df["keyword_len"] = df["keyword"].fillna("").astype(str).str.len()
df["has_location"] = df["location"].notna().astype(int)
df["judge_confidence"] = pd.to_numeric(df["choose_one:confidence"], errors="coerce").fillna(0.0)
df["trusted_judgments"] = pd.to_numeric(df["_trusted_judgments"], errors="coerce").fillna(0.0)

prepared_cols = [
    "text_len",
    "keyword_len",
    "has_location",
    "judge_confidence",
    "trusted_judgments",
    "label",
]
prepared_df = df[prepared_cols].copy()

artifacts_dir = TRAINING_ROOT / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)
prepared_dataset_rel_path = Path("artifacts") / "socialmedia_disaster_prepared.csv"
prepared_dataset_path = TRAINING_ROOT / prepared_dataset_rel_path
prepared_df.to_csv(prepared_dataset_path, index=False)

print("Prepared dataset saved (from repo root):", repo_rel(prepared_dataset_path))
print("Prepared shape:", prepared_df.shape)
print("Label distribution:")
print(prepared_df["label"].value_counts())
display(prepared_df.head(5))


## Step 3: Create config for this dataset
This config uses the prepared CSV and trains a Random Forest first.

In [ ]:
print("[Step 3] Creating run config...")

run_config = {
    "dataset": {
        "path": prepared_dataset_rel_path.as_posix(),
        "abnormal_path": "",
        "target_column": "label",
        "train_split": 0.7,
        "val_split": 0.15,
        "test_split": 0.15,
        "random_seed": 42,
        "stratify": True,
    },
    "preprocessing": {
        "missing_value_strategy": "mean",
        "normalization": "standard",
        "encoding": "none",
        "feature_selection": False,
    },
    "model": {
        "type": "random_forest",
        "hyperparameters": {"n_estimators": 120, "max_depth": 8},
    },
    "training": {
        "batch_size": 32,
        "epochs": 10,
        "learning_rate": 0.001,
        "verbose": True,
    },
    "output": {
        "path": "checkpoints",
        "log_path": "logs",
        "save_best_only": True,
        "checkpoint_prefix": "socialmedia",
    },
}

config_rel_path = Path("configs") / "socialmedia_disaster_config.json"
config_path = TRAINING_ROOT / config_rel_path
config_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print("Config saved (from repo root):", repo_rel(config_path))
print(json.dumps(run_config["model"], indent=2))


## Step 4: Run the pipeline end-to-end

In [ ]:
from src import run_training_pipeline

print("[Step 4] Running training pipeline...")
rf_result = run_training_pipeline(
    config_path=config_path,
    preprocessing_config_path=None,
    run_id="socialmedia_rf_notebook",
    save_checkpoint=False,
)

print("Run ID:", rf_result["run_id"])
print("Rows:", rf_result["rows"])
print("Validation metrics:")
print(json.dumps(rf_result["metrics"]["validation"], indent=2, default=str))
print("Test metrics:")
print(json.dumps(rf_result["metrics"]["test"], indent=2, default=str))


## Step 5: Try a PyTorch classification model (Simple MLP)
Reuse same data + config, then switch to a PyTorch model.


In [ ]:
import copy

print("[Step 5] Running PyTorch MLP for comparison...")

feature_count = prepared_df.drop(columns=["label"]).shape[1]
print("Detected input_dim for MLP:", feature_count)

torch_config = copy.deepcopy(run_config)
torch_config["model"] = {
    "type": "pytorch_mlp",
    "task_type": "classification",
    "hyperparameters": {
        "input_dim": int(feature_count),
        "hidden_dim": 64,
        "output_dim": 2,
    },
}
torch_config["training"]["epochs"] = 12

torch_config_rel_path = Path("configs") / "socialmedia_disaster_config_pytorch.json"
torch_config_path = TRAINING_ROOT / torch_config_rel_path
torch_config_path.write_text(json.dumps(torch_config, indent=2), encoding="utf-8")
print("PyTorch config saved (from repo root):", repo_rel(torch_config_path))

try:
    torch_result = run_training_pipeline(
        config_path=torch_config_path,
        preprocessing_config_path=None,
        run_id="socialmedia_pytorch_notebook",
        save_checkpoint=False,
    )
    print("Random Forest test accuracy:", rf_result["metrics"]["test"].get("accuracy"))
    print("PyTorch MLP test accuracy:", torch_result["metrics"]["test"].get("accuracy"))
except ImportError as exc:
    print("PyTorch run skipped (dependency issue):", exc)


## Step 6: Check logs and artifacts
These files help others reproduce the same run.

In [ ]:
print("[Step 6] Listing generated files...")
print("Artifacts:")
for p in sorted(artifacts_dir.glob("socialmedia_disaster_*")):
    print(" -", repo_rel(p))

print("\nConfigs:")
for p in sorted((TRAINING_ROOT / "configs").glob("*socialmedia*.json")):
    print(" -", repo_rel(p))

print("\nLatest logs:")
for p in sorted((TRAINING_ROOT / "logs").glob("*.log"))[-5:]:
    print(" -", repo_rel(p))


## How Others Can Reuse This

- Put a CSV in `ai-ml/datasets/`
- Update dataset path and target logic in **Step 2**
- Keep the same workflow (prepare -> config -> run -> compare)
- Re-run all cells from top to bottom
